In [8]:
# --- CELL 1: IMPORTS ---
import os
import random
import rasterio
import numpy as np
import pandas as pd
import shapely.wkt
import matplotlib.pyplot as plt
import cv2

# PyTorch Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# --- REVISED CELL: DYNAMIC BATCH DATA LOADING ---
import os
import glob
import rasterio
import numpy as np
import pandas as pd
import shapely.wkt
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Point to the DIRECTORIES, not individual files
BASE_DIR = '/kaggle/input/datasets/sandhiwangiyana/spacenet-6-multisensor-allweather-mapping' 
RGB_DIR = os.path.join(BASE_DIR, 'AOI_11_Rotterdam/PS-RGB')
SAR_DIR = os.path.join(BASE_DIR, 'AOI_11_Rotterdam/SAR-Intensity')
CSV_PATH = os.path.join(BASE_DIR, 'AOI_11_Rotterdam/SummaryData/SN6_Train_AOI_11_Rotterdam_Buildings.csv')

def load_optical(filepath):
    with rasterio.open(filepath) as src:
        img = src.read([1, 2, 3]) 
        img = np.moveaxis(img, 0, -1)
        max_val = np.max(img)
        if max_val > 0: img = img / max_val
    return img

def load_sar(filepath):
    with rasterio.open(filepath) as src:
        sar_img = src.read([1, 2, 3, 4]) 
        sar_img = np.moveaxis(sar_img, 0, -1) 
        p99 = np.percentile(sar_img, 99)
        sar_img = np.clip(sar_img, 0, p99)
        if p99 > 0: sar_img = sar_img / p99 
    return sar_img

def prepare_multi_image_dataset(rgb_dir, sar_dir, csv_path, num_images=10):
    """Scans the dataset directory and loads buildings from multiple distinct image tiles."""
    print(f"Scanning for {num_images} image tiles...")
    
    # Grab a random or sorted subset of the .tif files in the optical directory
    all_rgb_files = sorted(glob.glob(os.path.join(rgb_dir, '*.tif')))[:num_images]
    
    # Load the massive CSV into memory once
    full_df = pd.read_csv(csv_path)
    
    master_building_list = []
    
    for rgb_path in all_rgb_files:
        filename = os.path.basename(rgb_path)
        
        # Extract the exact tile ID (e.g., 'tile_8679') from the filename
        tile_str = 'tile_' + filename.split('_tile_')[-1].replace('.tif', '')
        
        # Construct the path to the matching SAR file
        sar_filename = filename.replace('PS-RGB', 'SAR-Intensity')
        sar_path = os.path.join(sar_dir, sar_filename)
        
        if not os.path.exists(sar_path):
            continue # Skip if the SAR equivalent is missing
            
        # Load the actual arrays for this specific tile
        opt_img = load_optical(rgb_path)
        sar_img = load_sar(sar_path)
        combined_img = np.dstack((opt_img, sar_img)) # 7 channels
        
        # Filter CSV for JUST this tile
        tile_buildings = full_df[full_df['ImageId'].str.contains(tile_str)].copy()
        
        for _, row in tile_buildings.iterrows():
            poly = shapely.wkt.loads(row['PolygonWKT_Pix'])
            height = row['Mean_Building_Height']
            if poly.is_empty or height <= 0:
                continue
                
            # KEY CHANGE: Store the building info ALONG WITH a reference to its specific parent image
            master_building_list.append({
                'parent_image': combined_img,
                'polygon': poly,
                'true_height': height
            })
            
    print(f"Successfully loaded {len(all_rgb_files)} images containing {len(master_building_list)} total buildings.")
    return master_building_list

# --- UPDATED DATASET CLASS ---
class SpaceNetBuildingDataset(Dataset):
    def __init__(self, master_building_list, crop_size=64):
        # We no longer pass a single image. We pass the list of all buildings + their parent images.
        self.buildings = master_building_list
        self.crop_size = crop_size

    def __len__(self):
        return len(self.buildings)

    def __getitem__(self, idx):
        b = self.buildings[idx]
        
        # Extract the specific 7-channel image that this building belongs to
        combined_img = b['parent_image'] 
        poly = b['polygon']
        height = b['true_height']
        
        minx, miny, maxx, maxy = poly.bounds
        
        buffer = 8
        minx, miny = max(0, int(minx)-buffer), max(0, int(miny)-buffer)
        maxx = min(combined_img.shape[1], int(maxx)+buffer)
        maxy = min(combined_img.shape[0], int(maxy)+buffer)
        
        crop = combined_img[miny:maxy, minx:maxx]
        
        if crop.shape[0] == 0 or crop.shape[1] == 0:
             crop = np.zeros((self.crop_size, self.crop_size, 7))
             
        crop_resized = cv2.resize(crop, (self.crop_size, self.crop_size))
        
        tensor_img = torch.tensor(crop_resized, dtype=torch.float32).permute(2, 0, 1)
        tensor_height = torch.tensor([height], dtype=torch.float32)
        
        return tensor_img, tensor_height

In [ ]:
# --- CELL 5: CNN ARCHITECTURE ---
class BuildingHeightCNN(nn.Module):
    def __init__(self):
        super(BuildingHeightCNN, self).__init__()
        # Input layer expects 7 channels
        self.features = nn.Sequential(
            nn.Conv2d(7, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), # Added batch norm for training stability
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  
        )
        
        self.regression_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.4), # Prevent overfitting
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1) # Final continuous output
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regression_head(x)
        return x

In [ ]:
# --- CELL 6: TRAINING PIPELINE ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on: {device}")

# 1. Initialize Data
print("Loading images and CSV into memory...")
optical_img = load_optical(OPTICAL_PATH)
sar_img = load_sar(SAR_PATH)
buildings = load_building_data(CSV_PATH, TILE_ID)

dataset = SpaceNetBuildingDataset(optical_img, sar_img, buildings, crop_size=CROP_SIZE)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# 2. Initialize Model
model = BuildingHeightCNN().to(device)
criterion = nn.MSELoss() 
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 3. Training Loop with Model Saving
print(f"Starting {EPOCHS}-epoch training loop...")
best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train() 
    running_loss = 0.0
    
    for batch_imgs, batch_heights in dataloader:
        batch_imgs, batch_heights = batch_imgs.to(device), batch_heights.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_imgs)
        loss = criterion(predictions, batch_heights)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(dataloader)
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] - Average MSE Loss: {avg_loss:.4f}")
    
    # --- CHECKPOINTING LOGIC ---
    # Only save the model if it performs better than previous epochs
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, BEST_MODEL_PATH)
        print(f" -> Checkpoint saved! New best loss: {best_loss:.4f}")

print("\nTraining Complete! Best model saved to Kaggle /working/ directory.")

In [ ]:
# --- CELL 7: TEST ON A RANDOM BUILDING ---
# 1. Load the best saved model weights
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval() # Strictly evaluation mode

# 2. Pick a random building from our dataset
random_idx = random.randint(0, len(dataset) - 1)
sample_tensor, true_height = dataset[random_idx]

# 3. Predict
with torch.no_grad():
    # Add the batch dimension [1, 7, 64, 64]
    input_tensor = sample_tensor.unsqueeze(0).to(device)
    predicted_height = model(input_tensor)

# 4. Extract just the optical part (first 3 channels) for visualization
# Note: Tensor is (C, H, W). We need (H, W, C) for Matplotlib
vis_img = sample_tensor[:3].permute(1, 2, 0).cpu().numpy()

# 5. Display the results
plt.figure(figsize=(6, 6))
plt.imshow(vis_img)
plt.title(f"True Height: {true_height.item():.1f}m | Predicted: {predicted_height.item():.1f}m")
plt.axis('off')
plt.show()

print(f"Detailed Results:")
print(f"Target Height: {true_height.item():.3f} meters")
print(f"Model Output:  {predicted_height.item():.3f} meters")
print(f"Error Margin:  {abs(true_height.item() - predicted_height.item()):.3f} meters")